# Loading ESM-C and testing it

https://huggingface.co/biohub/ESMC-300M

In [1]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

GFP = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"

# optionally use "biohub/ESMC-600M" or "biohub/ESMC-300M"
model = AutoModelForMaskedLM.from_pretrained("biohub/ESMC-6B", device_map="auto").eval()
tokenizer = AutoTokenizer.from_pretrained("biohub/ESMC-6B")

ESMC: transformer_engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically — measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.
ESMC: neither xformers nor flash-attn is installed; falling back to PyTorch ``F.scaled_dot_product_attention``. The attention reduction order in bf16 differs from a fused kernel by ~1 bf16 ULP per attention block; compounded across the 80-block stack this reaches ~O(100) max-diff on the unnormalized residual stream. Install xformers (preferred) with `pip install xformers` for a fused attention kernel.
ESMC: flash-attn rotary kernel not installed; falling back to pure-PyTorch RoPE. For faster GPU inference run `pip install flash-attn`.


🚨 No checkpoint found for ESMCForSequenceClassification.forward. Please add a `checkpoint` arg to `auto_docstring` or add one in ESMCConfig's docstring
🚨 No checkpoint found for ESMCForTokenClassification.forward. Please add a `checkpoint` arg to `auto_docstring` or add one in ESMCConfig's docstring


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [2]:
model

ESMCForMaskedLM(
  (esmc): ESMCModel(
    (embed): Embedding(64, 2560)
    (transformer): TransformerStack(
      (blocks): ModuleList(
        (0-79): 80 x UnifiedTransformerBlock(
          (attn): MultiHeadAttention(
            (layernorm_qkv): _PyTorchLayerNormLinear()
            (out_proj): Linear(in_features=2560, out_features=2560, bias=False)
            (q_ln): LayerNorm((2560,), eps=1e-05, elementwise_affine=True, bias=False)
            (k_ln): LayerNorm((2560,), eps=1e-05, elementwise_affine=True, bias=False)
            (rotary): RotaryEmbedding()
          )
          (ffn): _PyTorchLayerNormMLP()
        )
      )
      (norm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True, bias=False)
    )
    (_sae_models): ModuleDict()
  )
  (lm_head): Sequential(
    (0): Linear(in_features=2560, out_features=2560, bias=True)
    (1): GELU(approximate='none')
    (2): LayerNorm((2560,), eps=1e-05, elementwise_affine=True, bias=True)
    (3): Linear(in_features=2560, out_fe

```python
model = AutoModelForMaskedLM.from_pretrained("biohub/ESMC-300M", device_map="auto").eval()
tokenizer = AutoTokenizer.from_pretrained("biohub/ESMC-300M")
```

In [3]:
inputs = tokenizer(GFP, return_tensors="pt", padding=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.inference_mode():
    output = model(**inputs)

print(f"logits shape: {tuple(output.logits.shape)}")

logits shape: (1, 240, 64)


In [4]:
# output.logits: (batch, seq_len, vocab_size)
predicted_token_ids = output.logits.argmax(dim=-1)  # (batch, seq_len)

# Decode the full sequence
decoded = tokenizer.decode(predicted_token_ids[0], skip_special_tokens=True)
print(decoded)

K M S K K K K L F K G K V K V K V K G K G E I N G V K I K I K G K G E I D G K T G K L K L K G T D K D G K L P D D P P D G P K G L D G G L S L L S K L P E L L K D L D L F K K L L P E G T K L E K K L T L K E D G K G K L K L E L K L E G D K K I S K L E L K G I D I D K D G S L K G G K I S L K G G G G G L E L E A E K E E D G K K L D L K L K V N L K D G S I V I I L I L L G G I K L P D K L I L L K G G I L L L L L L F L S S D P N K K K D L L L L L G G L D K D G L L L L L D E L L K K


## Push the trained PlasticESM LoRA adapter to Hugging Face (`petadex`)

After running `python -m petabite.mlm_finetune` (Stage 1), the trained LoRA adapter +
tokenizer are saved to the trainer's `output_dir` (see `config/mlm.yaml`:
`outputs/esmc-mlm-lora-bs16-r32`). The cell below uploads that folder verbatim to the
**`petadex`** org — the same org the PETadex dataset was pushed to in
`Data-Processing.ipynb` (`petadex/catalytic-orfs-90pid`) — at the repo id declared in
the config (`push_to_hub.repo_id`: `petadex/esmc-300m-catalytic-lora`).

Run this on the AWS box where training produced `output_dir`. Authenticate first by
setting `HF_TOKEN` in the environment (a token with write access to `petadex`), e.g.
`export HF_TOKEN=hf_...`.

> **Note:** uploading the folder pushes exactly what `trainer.save_model` wrote — the
> PEFT `out_proj` adapter, tokenizer, `config.yaml`, and `requirements.txt`. If you
> trained with `lora_fused: true`, confirm the fused QKV/FFN LoRA weights are actually
> present in `output_dir` first — PEFT's adapter save only covers the `out_proj` LoRA.

In [ ]:
import os

from huggingface_hub import HfApi, login

# Mirrors config/mlm.yaml. Edit OUTPUT_DIR if you trained with a different output_dir.
OUTPUT_DIR = "../outputs/esmc-mlm-lora-bs16-r32"   # trainer output_dir (the trained adapter)
REPO_ID = "petadex/esmc-300m-catalytic-lora"       # push_to_hub.repo_id
PRIVATE = False                                     # push_to_hub.private

api = HfApi()
api.create_repo(REPO_ID, repo_type="model", private=PRIVATE, exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Upload PlasticESM (ESM-C 300M LoRA, MLM domain-adapted) adapter",
)
print(f"Pushed {OUTPUT_DIR} -> https://huggingface.co/{REPO_ID}")